# Run All Notebooks — SmarterDx Documentation Gap Analysis

This notebook executes all project notebooks in the correct order. Run this single notebook to reproduce the entire analysis from raw data to final charts.

**Prerequisites**: Raw data files must be present in `data/raw/` (NB01-NB03 download them automatically).

In [1]:
import subprocess
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
print(f'Project root: {PROJECT_ROOT}')

# Define all notebooks in execution order
notebooks = [
    ('Phase 1: Data Collection', [
        '1_data_collection/NB01_hospital_characteristics.ipynb',
        '1_data_collection/NB02_drg_cmi_data.ipynb',
        '1_data_collection/NB03_charges_payments.ipynb',
    ]),
    ('Phase 2: Hospital Analysis', [
        '2_hospital_analysis/NB04_data_cleaning_merge.ipynb',
        '2_hospital_analysis/NB05_feature_engineering.ipynb',
        '2_hospital_analysis/NB06_peer_group_analysis.ipynb',
    ]),
    ('Phase 3: Gap Modeling', [
        '3_gap_modeling/NB07_documentation_gap_scoring.ipynb',
        '3_gap_modeling/NB08_revenue_impact.ipynb',
    ]),
    ('Phase 4: Statistical Modeling', [
        '5_statistical_modeling/NB10a_additional_features.ipynb',
        '5_statistical_modeling/NB10_gap_drivers_model.ipynb',
    ]),
    ('Phase 5: Visualizations', [
        '4_visualizations/NB09_blog_charts.ipynb',
    ]),
]

# Flatten for counting
all_notebooks = [(phase, nb) for phase, nbs in notebooks for nb in nbs]
print(f'Total notebooks to run: {len(all_notebooks)}')
print()
for phase, nbs in notebooks:
    print(f'  {phase}:')
    for nb in nbs:
        print(f'    - {nb}')

Project root: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/notebooks
Total notebooks to run: 11

  Phase 1: Data Collection:
    - 1_data_collection/NB01_hospital_characteristics.ipynb
    - 1_data_collection/NB02_drg_cmi_data.ipynb
    - 1_data_collection/NB03_charges_payments.ipynb
  Phase 2: Hospital Analysis:
    - 2_hospital_analysis/NB04_data_cleaning_merge.ipynb
    - 2_hospital_analysis/NB05_feature_engineering.ipynb
    - 2_hospital_analysis/NB06_peer_group_analysis.ipynb
  Phase 3: Gap Modeling:
    - 3_gap_modeling/NB07_documentation_gap_scoring.ipynb
    - 3_gap_modeling/NB08_revenue_impact.ipynb
  Phase 4: Statistical Modeling:
    - 5_statistical_modeling/NB10a_additional_features.ipynb
    - 5_statistical_modeling/NB10_gap_drivers_model.ipynb
  Phase 5: Visualizations:
    - 4_visualizations/NB09_blog_charts.ipynb


## Execute All Notebooks

In [2]:
results = []
total_start = time.time()

for phase, nbs in notebooks:
    print(f'\n{"="*70}')
    print(f'{phase}')
    print(f'{"="*70}')
    
    for nb_path in nbs:
        full_path = PROJECT_ROOT / nb_path
        nb_dir = full_path.parent
        nb_name = full_path.name
        
        print(f'\n  Running: {nb_path}...', end=' ', flush=True)
        start = time.time()
        
        try:
            result = subprocess.run(
                ['jupyter', 'nbconvert', '--to', 'notebook', '--execute',
                 '--ExecutePreprocessor.timeout=600',
                 '--ExecutePreprocessor.kernel_name=python3',
                 '--output', nb_name,
                 str(full_path)],
                capture_output=True, text=True, timeout=660,
                cwd=str(nb_dir)
            )
            elapsed = time.time() - start
            
            if result.returncode == 0:
                status = 'SUCCESS'
                print(f'✓ ({elapsed:.1f}s)')
            else:
                status = 'FAILED'
                print(f'✗ ({elapsed:.1f}s)')
                # Print last few lines of stderr for debugging
                stderr_lines = result.stderr.strip().split('\n')
                for line in stderr_lines[-5:]:
                    print(f'    ERROR: {line}')
                    
        except subprocess.TimeoutExpired:
            elapsed = time.time() - start
            status = 'TIMEOUT'
            print(f'⏰ TIMEOUT ({elapsed:.1f}s)')
        except Exception as e:
            elapsed = time.time() - start
            status = f'ERROR: {str(e)}'
            print(f'✗ {status} ({elapsed:.1f}s)')
        
        results.append({
            'phase': phase,
            'notebook': nb_path,
            'status': status,
            'elapsed_seconds': round(elapsed, 1)
        })

total_elapsed = time.time() - total_start


Phase 1: Data Collection

  Running: 1_data_collection/NB01_hospital_characteristics.ipynb... ✓ (11.5s)

  Running: 1_data_collection/NB02_drg_cmi_data.ipynb... ✓ (10.4s)

  Running: 1_data_collection/NB03_charges_payments.ipynb... ✓ (8.4s)

Phase 2: Hospital Analysis

  Running: 2_hospital_analysis/NB04_data_cleaning_merge.ipynb... ✓ (5.9s)

  Running: 2_hospital_analysis/NB05_feature_engineering.ipynb... ✓ (5.8s)

  Running: 2_hospital_analysis/NB06_peer_group_analysis.ipynb... ✓ (7.6s)

Phase 3: Gap Modeling

  Running: 3_gap_modeling/NB07_documentation_gap_scoring.ipynb... ✓ (5.6s)

  Running: 3_gap_modeling/NB08_revenue_impact.ipynb... ✓ (5.2s)

Phase 4: Statistical Modeling

  Running: 5_statistical_modeling/NB10a_additional_features.ipynb... ✓ (13.0s)

  Running: 5_statistical_modeling/NB10_gap_drivers_model.ipynb... ✓ (6.8s)

Phase 5: Visualizations

  Running: 4_visualizations/NB09_blog_charts.ipynb... ✓ (8.4s)


## Results Summary

In [3]:
import pandas as pd

print('\n' + '='*70)
print('EXECUTION SUMMARY')
print('='*70)

df_results = pd.DataFrame(results)
print(f'\nTotal time: {total_elapsed:.1f}s ({total_elapsed/60:.1f} minutes)')
print(f'\nResults:')

success = (df_results['status'] == 'SUCCESS').sum()
failed = len(df_results) - success
print(f'  Succeeded: {success}/{len(df_results)}')
print(f'  Failed: {failed}/{len(df_results)}')

print(f'\nDetails:')
print(f'{"Notebook":<55s} {"Status":<10s} {"Time":>8s}')
print('-'*75)
for _, row in df_results.iterrows():
    nb_short = row['notebook']
    print(f'{nb_short:<55s} {row["status"]:<10s} {row["elapsed_seconds"]:>7.1f}s')

if failed > 0:
    print(f'\n⚠️  {failed} notebook(s) failed. Check error messages above.')
else:
    print(f'\n✓ All {success} notebooks executed successfully!')
    print(f'\nOutputs are in data/outputs/ subdirectories.')
    print(f'Blog charts are in data/outputs/blog_charts/')


EXECUTION SUMMARY

Total time: 88.6s (1.5 minutes)

Results:
  Succeeded: 11/11
  Failed: 0/11

Details:
Notebook                                                Status         Time
---------------------------------------------------------------------------
1_data_collection/NB01_hospital_characteristics.ipynb   SUCCESS       11.5s
1_data_collection/NB02_drg_cmi_data.ipynb               SUCCESS       10.4s
1_data_collection/NB03_charges_payments.ipynb           SUCCESS        8.4s
2_hospital_analysis/NB04_data_cleaning_merge.ipynb      SUCCESS        5.9s
2_hospital_analysis/NB05_feature_engineering.ipynb      SUCCESS        5.8s
2_hospital_analysis/NB06_peer_group_analysis.ipynb      SUCCESS        7.6s
3_gap_modeling/NB07_documentation_gap_scoring.ipynb     SUCCESS        5.6s
3_gap_modeling/NB08_revenue_impact.ipynb                SUCCESS        5.2s
5_statistical_modeling/NB10a_additional_features.ipynb  SUCCESS       13.0s
5_statistical_modeling/NB10_gap_drivers_model.ipynb     SU